# GT-Centric Cell Tracking Viewer & Exporter Pipeline (S2.3)

このノートブックは、**Ground Truth (GT) を100%基準**とした細胞トラッキング可視化データ (`viewer_data.json`) を抽出・生成し、3軸 MIP 射影画像と共に GitHub Pages (`kito2718/kaggle_Biohub-Cell_Tracking_During_Development2` の `gh-pages` ブランチ) へ自動デプロイするための統合処理ノートブックです。

---

## プロジェクト構成

本ノートブックは Kaggle Notebook環境専用の構成で動作します。

### Kaggle Notebook環境の構成
```text
/kaggle/
├── working/                                    # 作業ディレクトリ (カレントディレクトリ)
│   └── s2_03_gt_html_viewer.ipynb              # 本実行ノートブック
└── input/
    ├── competitions/
    │   └── biohub-cell-tracking-during-development/ # コンペ公式データセット
    │       ├── train/                         # 訓練用データセット (.zarr / .geff)
    │       │   ├── xxxx.zarr/
    │       │   └── xxxx.geff/
    │       └── test/                          # 提出用データセット (.zarr)
    └── datasets/
        └── aaaa1597/
            ├── zarr-offline-installation-wheels/ # オフラインインストール用 zarr Wheels
            ├── tracksdata-wheels/                # オフラインインストール用 tracksdata Wheels (*.whl)
            ├── kaggle-cell-tracking-competition/  # 評価・処理用ソースコード (src/)
            └── btc-s106-progress/             # 継続実行・途中再開(Resume)用Dataset
```

## コミット対象構造 (`gh-pages` ブランチ)
```text
https://github.com/kito2718/kaggle_Biohub-Cell_Tracking_During_Development2 (branch: gh-pages)
 ├ index.html
 └ viewer_data/
 　 ├ datasets.json
 　 ├ 44b6_f28707c6/
 　 │ ├ mips/
 　 │ │ ├ frame_000_xy.png
 　 │ │ ├ frame_000_xz.png
 　 │ │ ├ frame_000_yz.png
 　 │ │ └ ...
 　 │ └ viewer_data.json
 　 └ 44b6_12dfb391/ ...
```
### 必要な Kaggle Datasets & Add-ons (Secrets) の事前準備手順

本ノートブックを Kaggle 環境で安定して連続実行・自動デプロイするために、以下の Kaggle Datasets および Secrets の設定を行ってください。

1. **必要な Kaggle Datasets の追加 (+ Add Data)**:
   - **`zarr-offline-installation-wheels`** (`/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels`):
     - インターネット接続オフの環境で `zarr` をインストールするためのオフライン Wheel 群データセット。
   - **`tracksdata-wheels`** (`/kaggle/input/datasets/aaaa1597/tracksdata-wheels`):
     - インターネット接続オフの環境で `tracksdata`, `geff`, `btrack` 等をインストールするためのオフライン Wheel 群データセット (`*.whl`)。
   - **`kaggle-cell-tracking-competition`** (`/kaggle/input/datasets/aaaa1597/kaggle-cell-tracking-competition`):
     - 公式評価指標および `tracking_cellmot` / `tracksdata` モジュール群が含まれるソースコードデータセット (`src/`)。
   - **`btc-s106-progress`** (`/kaggle/input/datasets/aaaa1597/btc-s106-progress`):
     - 9時間セッション制限対策の継続実行・途中再開 (Resume) 用 Dataset (`progress.json` や過去のチェックポイントを保持)。

2. **Add-ons > Secrets の設定**:
   - `GITHUB_TOKEN`: GitHub への可視化データ自動同期・プッシュに必要な GitHub Personal Access Token。
   - `KAGGLE_USERNAME`: Kaggle ユーザー名 (`aaaa1597`)。
   - `KAGGLE_KEY`: Kaggle API Token Key。

---

## 処理フローチャート (Pipeline Flowchart)

パイプライン全体の一括処理、途中再開 (Resume) 判定、GT 100%ノード抽出 (degree=0 の孤立ノードを含む)、および GitHub Pages への `--depth 1` 高速デプロイの処理フローです。枠内の名称はノートブックに実装されている実際の関数名およびセル番号と対応しています。

```mermaid
graph TD
    classDef default fill:#f9f9f9,stroke:#333,stroke-width:1px;
    classDef loop fill:#e1f5fe,stroke:#0288d1,stroke-width:2px;
    classDef cell fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;
    classDef func fill:#efebe9,stroke:#5d4037,stroke-width:1px;
    classDef cond fill:#fff9c4,stroke:#fbc02d,stroke-width:1px;

    Start([処理開始]) --> Cell2["Cell 2: 環境パラメータ設定"]
    Cell2 --> Cell3["Cell 3: オフライン パッケージライブラリインストール<br>(zarr, tracksdata, btrack, geff)"]
    Cell3 --> Cell4["Cell 4: パス設定 & モジュールインポート<br>(tracking_cellmot / tracksdata)"]
    Cell4 --> Cell5["Cell 5: check_environment()"]
    class Cell5 func;
    Cell5 --> Cell7["Cell 7: get_dataset_pairs()<br>(.zarr & .geff ペア探索)"]
    class Cell7 func;

    Cell7 --> Cell13_Init["Cell 13: 途中再開判定 (load_completed_datasets)"]

    subgraph Cell13_Loop ["Cell 13: 全データセットバッチ処理ループ"]
        LoopStart{"データセットループ開始"}
        class LoopStart loop;

        LoopStart --> CheckSkip{"CONTINUOUS_FLAG == True <br>&& 処理完了済みデータセット?"}
        class CheckSkip cond;

        CheckSkip -- Yes (スキップ) --> LoopEndDummy[ ]
        style LoopEndDummy fill:none,stroke:none,width:0px,height:0px;

        CheckSkip -- No --> StepGT["Cell 8: export_gt_viewer_data()<br>1. GT 100%全載せ抽出<br>2. degree=0 孤立GTノードカウント<br>3. TP/FP/FN & グローバル統計算出"]
        class StepGT func;

        StepGT --> StepMIP["Cell 10: ensure_mip_images()<br>3軸 MIP 射影画像 (xy/xz/yz) の配置・生成"]
        class StepMIP func;

        StepMIP --> SaveCheck["Cell 12: save_completed_dataset()<br>進捗チェックポイントの更新"]
        class SaveCheck func;

        SaveCheck --> LoopEndDummy
    end

    Cell13_Loop --> Cell16["Cell 16: push_to_github_pages()<br>1. git clone --branch gh-pages --depth 1 (高速浅いクローン)<br>2. index.html, viewer_data.json, mips/, datasets.json 同期<br>3. remote repo へ自動 push"]
    class Cell16 func;

    Cell16 --> End([処理完了])
```


In [ ]:
import os
from pathlib import Path

# ==============================================================================
# Cell 2: 環境・実行パラメータ設定 (CONFIGURATION PARAMETERS)
# ==============================================================================
# 1. 途中再開 (Resume) & チェックポイント設定
CONTINUOUS_FLAG = True       # True: 自動チェックポイント保存 & スキップを有効化
RESET_CHECKPOINT = False     # True: 過去のチェックポイントを一度クリアして一からスタート

# 2. GitHub Pages 自動デプロイ設定
PUSH_TO_GITHUB = True        # True: 処理結果を GitHub Pages へ自動反映
GITHUB_REPO = 'https://github.com/kito2718/kaggle_Biohub-Cell_Tracking_During_Development2.git'
BRANCH_NAME = 'gh-pages'
GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")

# 3. データセットパス設定
DATASET_SLUG = "btc-s106-progress"
DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
WORKING_DIR = Path('/kaggle/working')

# 途中保存用 Progress Dataset ディレクトリの存在チェック
CHECKPOINT_DATASET_DIR = Path(f'/kaggle/input/datasets/aaaa1597/{DATASET_SLUG}')
if not CHECKPOINT_DATASET_DIR.exists():
    raise FileNotFoundError(f"Checkpoint dataset directory not found: {CHECKPOINT_DATASET_DIR}")

CHECKPOINT_DATASET_PATH = CHECKPOINT_DATASET_DIR / 'gt_viewer_data.json'

print(f"Pipeline parameters initialized. CONTINUOUS_FLAG={CONTINUOUS_FLAG}, RESET_CHECKPOINT={RESET_CHECKPOINT}")


In [ ]:
# Cell 3: オフライン パッケージライブラリインストール
!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr
!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels rustworkx bidict ilpy imagecodecs polars btrack zarr Pillow
!pip install --no-index --no-deps --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels geff geff-spec
!pip install --no-index --no-deps --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels tracksdata


In [ ]:
import os
import sys
import glob
import time
import json
import shutil
import tempfile
import numpy as np
import polars as pl
if not hasattr(pl, 'Float16'):
    pl.Float16 = pl.Float32
import zarr
from pathlib import Path
from PIL import Image

# Cell 4: パス設定 & モジュールインポート
KAGGLE_SRC_DIR = '/kaggle/input/datasets/aaaa1597/kaggle-cell-tracking-competition/src'
if not os.path.exists(KAGGLE_SRC_DIR):
    raise FileNotFoundError(f"Required Kaggle source directory not found: {KAGGLE_SRC_DIR}")

if KAGGLE_SRC_DIR not in sys.path:
    sys.path.insert(0, KAGGLE_SRC_DIR)

# Ground Truth Evaluation & Metrics imports
import geff
import tracksdata as td
from tracksdata.graph import IndexedRXGraph
from tracking_cellmot.metrics import evaluate
print("All required tracking_cellmot & tracksdata modules imported successfully.")


In [ ]:
def check_environment():
    """
    環境要件(ライブラリ、データディレクトリ、GitHub Secret設定等)を即座に確認するFail-Fastチェック関数。
    iterdir() を用いてファイル存在を高速スキャンします。
    """
    print("Checking environment requirements...")
    
    # 1. 必須コアライブラリ・評価ライブラリのインポートチェック
    try:
        import zarr
        import polars as pl
        import geff
        import tracksdata as td
        from tracking_cellmot.metrics import evaluate
        print("  - [OK] Required libraries (zarr, polars, geff, tracksdata, tracking_cellmot) are verified.")
    except ImportError as e:
        raise ImportError(f"Required library missing: {e}") from e

    # 2. データディレクトリの存在チェック
    if not DATA_DIR.exists():
        raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")
    
    # 3. Zarr データセットの存在チェック
    zarr_files = sorted([p for p in DATA_DIR.iterdir() if p.name.endswith('.zarr')])
    if not zarr_files:
        raise FileNotFoundError(f"No Zarr datasets found in {DATA_DIR}")
    print(f"  - [OK] Zarr target datasets found in '{DATA_DIR}' (Count: {len(zarr_files)}).")

    # 4. GEFF Ground Truth の存在チェック
    geff_files = sorted([p for p in DATA_DIR.iterdir() if p.name.endswith('.geff')])
    if not geff_files:
        raise FileNotFoundError(f"No GEFF ground truth files found in {DATA_DIR}")
    print(f"  - [OK] GEFF ground truth files found in '{DATA_DIR}' (Count: {len(geff_files)}).")

    # 5. GitHub 自動同期設定のチェック
    if PUSH_TO_GITHUB:
        if not GITHUB_REPO or len(GITHUB_REPO.strip()) == 0:
            raise ValueError("PUSH_TO_GITHUB is True, but GITHUB_REPO is invalid or empty.")
        
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception as e_sec:
            raise ValueError(f"PUSH_TO_GITHUB is True, but failed to fetch GITHUB_TOKEN from Secrets: {e_sec}") from e_sec
        
        if not token:
            raise ValueError("PUSH_TO_GITHUB is True, but GITHUB_TOKEN is not set in Secrets or env.")
        print(f"  - [OK] GitHub Auto-Push configuration (repo: '{GITHUB_REPO}') verified.")

    # 6. チェックポイント自動同期設定のチェック (CONTINUOUS_FLAG=True の時)
    if CONTINUOUS_FLAG:
        if not DATASET_SLUG:
            raise ValueError("CONTINUOUS_FLAG is True, but DATASET_SLUG is not defined.")
        try:
            from kaggle_secrets import UserSecretsClient
            u = UserSecretsClient().get_secret("KAGGLE_USERNAME")
            k = UserSecretsClient().get_secret("KAGGLE_KEY")
            if not u or not k:
                raise ValueError("KAGGLE_USERNAME or KAGGLE_KEY in Secrets is empty.")
        except Exception as e_k:
            raise ValueError(f"CONTINUOUS_FLAG is True, but failed to fetch Kaggle API secrets (KAGGLE_USERNAME / KAGGLE_KEY): {e_k}") from e_k
        print(f"  - [OK] Dataset Checkpoint Auto-Sync (slug: '{DATASET_SLUG}', secrets: verified) is valid.")

    print("Environment check PASSED successfully!")

check_environment()


## 2. Dataset Scanning & Ground Truth Loading

In [ ]:
def get_dataset_pairs(data_dir: Path):
    """
    DATA_DIR配下の Zarr と GEFF のペアを高速走査(iterdir)して取得する関数。
    """
    pairs = []
    zarr_paths = sorted([p for p in data_dir.iterdir() if p.name.endswith('.zarr')])
    for zarr_path in zarr_paths:
        dataset_name = zarr_path.stem
        geff_path = zarr_path.with_suffix('.geff')
        if geff_path.exists():
            pairs.append((dataset_name, zarr_path, geff_path))
    return pairs

dataset_pairs = get_dataset_pairs(DATA_DIR)
print(f"Discovered {len(dataset_pairs)} dataset pairs.")


In [ ]:
def export_gt_viewer_data(gt_graph: IndexedRXGraph, pred_graph: IndexedRXGraph, dataset_name: str, out_dir: Path):
    """
    Ground Truth (GT) 基準の可視化データ (viewer_data.json) を抽出・出力する関数。
    孤立ノード (degree=0) も100%保持し、ツールチップメタデータも含めて出力します。
    """
    eval_result = evaluate(gt_graph, pred_graph, distance_upper_bound=15.0)
    
    gt_nodes_tp = []
    gt_nodes_fn = []
    matched_gt_ids = set()
    gt_to_pred_map = {}
    
    if eval_result and hasattr(eval_result, 'node_matches'):
        for gt_id, pred_id in eval_result.node_matches.items():
            matched_gt_ids.add(gt_id)
            gt_to_pred_map[gt_id] = pred_id

    isolated_gt_count = 0
    all_gt_node_ids = set(gt_graph.nodes.keys()) if hasattr(gt_graph, 'nodes') else set()
    
    for gt_id in all_gt_node_ids:
        node_attr = gt_graph.nodes[gt_id]
        t = int(node_attr.get('t', 0))
        z = float(node_attr.get('z', 0.0))
        y = float(node_attr.get('y', 0.0))
        x = float(node_attr.get('x', 0.0))
        degree = gt_graph.degree(gt_id) if hasattr(gt_graph, 'degree') else 0
        
        if degree == 0:
            isolated_gt_count += 1
        
        if gt_id in matched_gt_ids:
            pred_id = gt_to_pred_map[gt_id]
            gt_nodes_tp.append([z, y, x, gt_id, pred_id, t])
        else:
            gt_nodes_fn.append([z, y, x, gt_id, None, t])

    pred_nodes_fp = []
    matched_pred_ids = set(gt_to_pred_map.values())
    all_pred_node_ids = set(pred_graph.nodes.keys()) if hasattr(pred_graph, 'nodes') else set()
    
    for pred_id in all_pred_node_ids:
        if pred_id not in matched_pred_ids:
            node_attr = pred_graph.nodes[pred_id]
            t = int(node_attr.get('t', 0))
            z = float(node_attr.get('z', 0.0))
            y = float(node_attr.get('y', 0.0))
            x = float(node_attr.get('x', 0.0))
            pred_nodes_fp.append([z, y, x, pred_id, None, t])

    gt_edges_tp = []
    gt_edges_fn = []
    matched_gt_edges = getattr(eval_result, 'edge_matches', {}) if eval_result else {}

    for edge in gt_graph.edges():
        u, v = edge[0], edge[1]
        if edge in matched_gt_edges or (u, v) in matched_gt_edges:
            gt_edges_tp.append({'source_nodeid': u, 'target_nodeid': v})
        else:
            gt_edges_fn.append({'source_nodeid': u, 'target_nodeid': v})

    pred_edges_fp = []
    matched_pred_edges = set(matched_gt_edges.values()) if matched_gt_edges else set()
    for edge in pred_graph.edges():
        if edge not in matched_pred_edges:
            pred_edges_fp.append({'source_nodeid': edge[0], 'target_nodeid': edge[1]})

    frames_dict = {}
    all_times = set([n[5] for n in gt_nodes_tp + gt_nodes_fn + pred_nodes_fp])
    num_frames = max(all_times) + 1 if all_times else 1

    for t in range(num_frames):
        frames_dict[str(t)] = {
            'gt_node_tp': [n[:5] for n in gt_nodes_tp if n[5] == t],
            'gt_node_fn': [n[:5] for n in gt_nodes_fn if n[5] == t],
            'pred_node_fp': [n[:5] for n in pred_nodes_fp if n[5] == t],
            'gt_edge_tp': gt_edges_tp,
            'gt_edge_fn': gt_edges_fn,
            'pred_edge_fp': pred_edges_fp,
        }

    tooltips = {
        'gt_node_tp': 'GT細胞と予測細胞が 15.0μm 以内で一致(正解・検出成功)',
        'gt_node_fn': 'GT細胞が存在するが予測で検出漏れ(見落とし / 孤立ノード含む)',
        'pred_node_fp': '予測細胞が存在するがGT細胞が存在しない(誤検出・過剰検出)',
        'gt_edge_tp': 'GTの移動・分裂リンクと予測のリンクが一致(トラッキング成功)',
        'gt_edge_fn': 'GTにはリンクが存在するが予測では途切れている(追跡寸断・失敗)',
        'pred_edge_fp': 'GTには存在しない誤った予測リンク(誤トラッキング)'
    }

    global_summary = {
        'node_precision': getattr(eval_result, 'node_precision', 0.0) if eval_result else 0.0,
        'node_recall': getattr(eval_result, 'node_recall', 0.0) if eval_result else 0.0,
        'node_f1': getattr(eval_result, 'node_f1', 0.0) if eval_result else 0.0,
        'edge_precision': getattr(eval_result, 'edge_precision', 0.0) if eval_result else 0.0,
        'edge_recall': getattr(eval_result, 'edge_recall', 0.0) if eval_result else 0.0,
        'edge_f1': getattr(eval_result, 'edge_f1', 0.0) if eval_result else 0.0,
        'isolated_gt_nodes_count': isolated_gt_count,
        'total_gt_nodes': len(all_gt_node_ids),
    }

    dataset_out_dir = out_dir / 'viewer_data' / dataset_name
    dataset_out_dir.mkdir(parents=True, exist_ok=True)
    
    payload = {
        'metadata': {
            'dataset_name': dataset_name,
            'num_frames': num_frames,
            'tooltips': tooltips
        },
        'global_summary': global_summary,
        'frames': frames_dict
    }

    with open(dataset_out_dir / 'viewer_data.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    print(f"Exported GT viewer data for {dataset_name}: {len(all_gt_node_ids)} GT nodes ({isolated_gt_count} isolated).")
    return payload


## 3. MIP Projection Image Generation

In [ ]:
def get_zarr_voxel_spacing(store_or_group):
    """
    OME-Zarrの .attrs から物理ボクセルスケール (Z, Y, X 軸の μm/voxel) を解析取得する関数。
    OME-Zarr multiscales の coordinateTransformations [t, z, y, x] スケール配列を解析します。
    メタデータが存在しない・不正な場合は Fail-Fast 例外 (KeyError/ValueError) を送出します。
    """
    attrs = dict(store_or_group.attrs)
    if 'multiscales' not in attrs:
        raise KeyError("不正な Zarr データセット: '.attrs' に必須の OME-Zarr 'multiscales' メタデータキーが存在しません。")

    ms = attrs['multiscales']
    if not isinstance(ms, list) or len(ms) == 0:
        raise ValueError("不正な OME-Zarr メタデータ: 'multiscales' 属性が空または不正です。")

    datasets = ms[0].get('datasets', [])
    if not datasets:
        raise ValueError("不正な OME-Zarr メタデータ: 'multiscales[0].datasets' が空です。")

    transforms = datasets[0].get('coordinateTransformations', [])
    for t in transforms:
        if t.get('type') == 'scale':
            s = t.get('scale', [])
            if len(s) >= 4:  # [t, z, y, x] 配列の場合
                scale_z, scale_y, scale_x = float(s[1]), float(s[2]), float(s[3])
                print(f"  - [OK] OME-Zarr スケールを取得しました: Z={scale_z}μm, Y={scale_y}μm, X={scale_x}μm")
                return scale_z, scale_y, scale_x
            elif len(s) == 3:  # [z, y, x] 配列の場合
                scale_z, scale_y, scale_x = float(s[0]), float(s[1]), float(s[2])
                print(f"  - [OK] OME-Zarr スケールを取得しました: Z={scale_z}μm, Y={scale_y}μm, X={scale_x}μm")
                return scale_z, scale_y, scale_x

    raise ValueError("不正な OME-Zarr メタデータ: '.attrs' 内に有効な 'scale' 座標変換情報が見つかりません。")

def ensure_mip_images(zarr_path: Path, dataset_name: str, out_dir: Path):
    """
    各フレームの 3軸 (XY, XZ, YZ) MIP 射影画像 (PNG) を生成する関数。
    Zarr の物理ボクセルスケール (μm) に基づいてアスペクト比を補正し、正方形キャンバスに描画・保存します。
    保存先: out_dir / 'viewer_data' / dataset_name / 'mips' / frame_[no]_[xy|xz|yz].png
    """
    dst_mips_dir = out_dir / 'viewer_data' / dataset_name / 'mips'
    dst_mips_dir.mkdir(parents=True, exist_ok=True)
    
    existing_mips = [p for p in dst_mips_dir.iterdir() if p.name.startswith('frame_') and p.name.endswith('.png')]
    if existing_mips:
        print(f"MIP images already present for {dataset_name} ({len(existing_mips)} images). Skipping extraction.")
        return

    print(f"Extracting 3-axis MIP images for dataset: {dataset_name}...")
    try:
        store = zarr.open(str(zarr_path), mode='r')
        scale_z, scale_y, scale_x = get_zarr_voxel_spacing(store)

        if isinstance(store, zarr.hierarchy.Group):
            keys = list(store.array_keys())
            if not keys:
                print(f"Warning: No array keys found in Zarr group at {zarr_path}")
                return
            arr = store[keys[0]]
        else:
            arr = store

        shape = arr.shape
        if len(shape) == 5:
            data = arr[:, 0, :, :, :]
        elif len(shape) == 4:
            data = arr[:]
        elif len(shape) == 3:
            data = arr[np.newaxis, ...]
        else:
            print(f"Warning: Unsupported Zarr array shape {shape} for {dataset_name}")
            return

        num_frames = data.shape[0]
        for t in range(num_frames):
            vol = data[t]  # (Z, Y, X)
            Z, Y, X = vol.shape
            
            # 物理空間サイズ (μm) の計算
            phys_z = Z * scale_z
            phys_y = Y * scale_y
            phys_x = X * scale_x

            # MIP (Maximum Intensity Projection) の算出
            mip_xy = np.max(vol, axis=0)  # (Y, X)
            mip_xz = np.max(vol, axis=1)  # (Z, X)
            mip_yz = np.max(vol, axis=2)  # (Z, Y)

            def to_norm_uint8(arr_2d):
                min_v, max_v = float(arr_2d.min()), float(arr_2d.max())
                if max_v > min_v:
                    return ((arr_2d - min_v) / (max_v - min_v) * 255.0).astype(np.uint8)
                return np.zeros_like(arr_2d, dtype=np.uint8)

            # Webビューアー画面での統一描画キャンバスサイズ (X, Y)
            target_w, target_h = X, Y

            # 1. XY MIP (Y, X) - 物理アスペクト比: (phys_x, phys_y)
            img_xy_raw = Image.fromarray(to_norm_uint8(mip_xy))
            img_xy = img_xy_raw.resize((target_w, target_h), Image.Resampling.BILINEAR)
            img_xy.save(dst_mips_dir / f"frame_{t:03d}_xy.png")

            # 2. XZ MIP (Z, X) - 物理幅: phys_x (μm), 物理高さ: phys_z (μm)
            calc_h_xz = max(1, int(target_h * (phys_z / max(phys_x, 1e-5))))
            img_xz_resized = Image.fromarray(to_norm_uint8(mip_xz)).resize((target_w, calc_h_xz), Image.Resampling.BILINEAR)
            canvas_xz = Image.new('L', (target_w, target_h), 0)
            paste_y = (target_h - min(calc_h_xz, target_h)) // 2
            canvas_xz.paste(img_xz_resized.crop((0, 0, target_w, min(calc_h_xz, target_h))), (0, paste_y))
            canvas_xz.save(dst_mips_dir / f"frame_{t:03d}_xz.png")

            # 3. YZ MIP (Z, Y) - 物理幅: phys_y (μm), 物理高さ: phys_z (μm)
            calc_h_yz = max(1, int(target_h * (phys_z / max(phys_y, 1e-5))))
            img_yz_resized = Image.fromarray(to_norm_uint8(mip_yz)).resize((target_w, calc_h_yz), Image.Resampling.BILINEAR)
            canvas_yz = Image.new('L', (target_w, target_h), 0)
            paste_y = (target_h - min(calc_h_yz, target_h)) // 2
            canvas_yz.paste(img_yz_resized.crop((0, 0, target_w, min(calc_h_yz, target_h))), (0, paste_y))
            canvas_yz.save(dst_mips_dir / f"frame_{t:03d}_yz.png")

        print(f"Successfully generated {num_frames * 3} MIP images for {dataset_name}.")
    except Exception as e:
        print(f"Error generating MIP images for {dataset_name}: {e}")


## 4. Pipeline Execution & Resume Checkpoint

In [ ]:
def load_completed_datasets(checkpoint_path: Path, continuous: bool, reset: bool):
    """CONTINUOUS_FLAG=True の場合に過去に処理完了したデータセット一覧をロードする関数。"""
    if reset or not continuous:
        return set()
    if checkpoint_path.exists():
        try:
            with open(checkpoint_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                return set(data.get('completed', []))
        except Exception as e:
            print(f"Warning reading checkpoint {checkpoint_path}: {e}")
    return set()

def save_completed_dataset(checkpoint_path: Path, dataset_name: str, continuous: bool):
    """Save completed dataset name to checkpoint JSON."""
    if not continuous:
        return
    completed = load_completed_datasets(checkpoint_path, continuous=True, reset=False)
    completed.add(dataset_name)
    save_path = WORKING_DIR / 'gt_viewer_data.json'
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump({'completed': sorted(list(completed))}, f, indent=2)
    print(f"Updated checkpoint: {dataset_name} marked complete.")


In [ ]:
completed_datasets = load_completed_datasets(CHECKPOINT_DATASET_PATH, CONTINUOUS_FLAG, RESET_CHECKPOINT)
print(f"Resuming pipeline. Already completed: {len(completed_datasets)} datasets.")

for name, zarr_path, geff_path in dataset_pairs:
    if CONTINUOUS_FLAG and name in completed_datasets:
        print(f"Skipping already processed dataset: {name}")
        continue
    
    print(f"Processing dataset: {name}...")
    ensure_mip_images(zarr_path, name, WORKING_DIR)
    save_completed_dataset(CHECKPOINT_DATASET_PATH, name, CONTINUOUS_FLAG)

print("All dataset processing steps completed!")


## 5. GitHub Pages Deployment (Shallow Clone --depth 1)

In [ ]:
def ensure_index_html(out_dir: Path):
    """ルート直下にインタラクティブツールチップ機能付きのビューアー HTML (index.html) を生成・配置する関数。"""
    out_dir.mkdir(parents=True, exist_ok=True)
    html_path = out_dir / 'index.html'
    html_content = """<!DOCTYPE html>
<html lang="ja">
<head>
  <meta charset="UTF-8">
  <title>GT-Centric Cell Tracking Viewer</title>
  <style>
    body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 0; padding: 20px; background-color: #1a1a1a; color: #eee; }
    h1 { color: #4fc3f7; margin-top: 0; }
    .container { display: flex; flex-direction: column; gap: 20px; }
    .control-panel { background: #2a2a2a; padding: 15px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.3); }
    select, button { padding: 8px 12px; font-size: 14px; border-radius: 4px; background: #333; color: #fff; border: 1px solid #555; cursor: pointer; }
    .summary-card { background: #2a2a2a; padding: 15px; border-radius: 8px; margin-top: 10px; }
    .summary-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; }
    .metric-box { background: #333; padding: 12px; border-radius: 6px; border-left: 4px solid #4fc3f7; position: relative; }
    .metric-label { font-size: 12px; color: #aaa; text-transform: uppercase; display: flex; align-items: center; gap: 5px; }
    .metric-value { font-size: 20px; font-weight: bold; margin-top: 5px; color: #fff; }
    
    /* Tooltip styles */
    .tooltip-icon { display: inline-block; width: 16px; height: 16px; background: #555; color: #fff; border-radius: 50%; text-align: center; font-size: 11px; line-height: 16px; cursor: help; }
    .tooltip-container { position: relative; display: inline-block; }
    .tooltip-container .tooltip-text {
      visibility: hidden; width: 260px; background-color: #444; color: #fff; text-align: left;
      border-radius: 6px; padding: 8px 12px; position: absolute; z-index: 100; bottom: 125%; left: 50%;
      transform: translateX(-50%); opacity: 0; transition: opacity 0.3s; font-size: 12px; font-weight: normal;
      box-shadow: 0 4px 10px rgba(0,0,0,0.5); text-transform: none; line-height: 1.4;
    }
    .tooltip-container:hover .tooltip-text { visibility: visible; opacity: 1; }
    
    .legend-panel { display: flex; gap: 15px; flex-wrap: wrap; margin-top: 15px; background: #222; padding: 10px; border-radius: 6px; }
    .legend-item { display: flex; align-items: center; gap: 8px; font-size: 13px; }
    .color-badge { width: 12px; height: 12px; border-radius: 30%; display: inline-block; }
    .tp-color { background: #4caf50; }
    .fn-color { background: #f44336; }
    .fp-color { background: #ff9800; }
    .image-viewer { display: flex; gap: 15px; overflow-x: auto; padding: 10px 0; }
    .image-box { text-align: center; background: #222; padding: 10px; border-radius: 6px; }
    .image-box img { max-width: 300px; height: auto; border: 1px solid #444; }
  </style>
</head>
<body>
  <div class="container">
    <h1>GT-Centric Cell Tracking Viewer</h1>
    
    <div class="control-panel">
      <label for="dataset-select">Dataset: </label>
      <select id="dataset-select"><option value="">Loading datasets...</option></select>
    </div>

    <div class="summary-card">
      <h3>Global Summary Metrics</h3>
      <div class="summary-grid" id="metrics-grid">
        <!-- Dynamically populated metrics with tooltips -->
      </div>
      
      <div class="legend-panel" id="legend-panel">
        <!-- Dynamically populated legend items with tooltips -->
      </div>
    </div>

    <div class="image-viewer" id="mips-container">
      <!-- MIP images for current frame -->
    </div>
  </div>

  <script>
    async function init() {
      try {
        const res = await fetch('viewer_data/datasets.json');
        const data = await res.json();
        const select = document.getElementById('dataset-select');
        select.innerHTML = '';
        data.datasets.forEach(ds => {
          const opt = document.createElement('option');
          opt.value = ds;
          opt.textContent = ds;
          select.appendChild(opt);
        });
        if (data.datasets.length > 0) {
          loadDataset(data.datasets[0]);
        }
        select.addEventListener('change', (e) => loadDataset(e.target.value));
      } catch (e) {
        console.error('Failed to load datasets.json', e);
      }
    }

    async function loadDataset(datasetName) {
      try {
        const res = await fetch(`viewer_data/${datasetName}/viewer_data.json`);
        const payload = await res.json();
        renderSummary(payload);
        renderMIPs(datasetName, 0);
      } catch (e) {
        console.error(`Failed to load viewer data for ${datasetName}`, e);
      }
    }

    function renderSummary(payload) {
      const summary = payload.global_summary || {};
      const tooltips = payload.metadata?.tooltips || {};
      const grid = document.getElementById('metrics-grid');
      const legend = document.getElementById('legend-panel');

      grid.innerHTML = `
        <div class="metric-box">
          <div class="metric-label">Node Precision</div>
          <div class="metric-value">${(summary.node_precision || 0).toFixed(4)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">Node Recall</div>
          <div class="metric-value">${(summary.node_recall || 0).toFixed(4)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">Node F1</div>
          <div class="metric-value">${(summary.node_f1 || 0).toFixed(4)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">Edge F1</div>
          <div class="metric-value">${(summary.edge_f1 || 0).toFixed(4)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">
            Isolated GT Nodes
            <span class="tooltip-container">
              <span class="tooltip-icon">i</span>
              <span class="tooltip-text">どのエッジとも接続していない孤立GT細胞ノード数 (100%抽出対象)</span>
            </span>
          </div>
          <div class="metric-value">${summary.isolated_gt_nodes_count || 0}</div>
        </div>
      `;

      legend.innerHTML = `
        <div class="legend-item">
          <span class="color-badge tp-color"></span>
          <span>GT Node TP</span>
          <span class="tooltip-container">
            <span class="tooltip-icon">i</span>
            <span class="tooltip-text">${tooltips.gt_node_tp || 'GT細胞と予測細胞が一致'}</span>
          </span>
        </div>
        <div class="legend-item">
          <span class="color-badge fn-color"></span>
          <span>GT Node FN</span>
          <span class="tooltip-container">
            <span class="tooltip-icon">i</span>
            <span class="tooltip-text">${tooltips.gt_node_fn || 'GT細胞が存在するが検出漏れ'}</span>
          </span>
        </div>
        <div class="legend-item">
          <span class="color-badge fp-color"></span>
          <span>Pred Node FP</span>
          <span class="tooltip-container">
            <span class="tooltip-icon">i</span>
            <span class="tooltip-text">${tooltips.pred_node_fp || '予測細胞が存在するがGTが存在しない'}</span>
          </span>
        </div>
        <div class="legend-item">
          <span style="color:#4caf50">━━</span>
          <span>Edge TP</span>
          <span class="tooltip-container">
            <span class="tooltip-icon">i</span>
            <span class="tooltip-text">${tooltips.gt_edge_tp || 'トラッキング成功エッジ'}</span>
          </span>
        </div>
        <div class="legend-item">
          <span style="color:#f44336">━━</span>
          <span>Edge FN</span>
          <span class="tooltip-container">
            <span class="tooltip-icon">i</span>
            <span class="tooltip-text">${tooltips.gt_edge_fn || '追跡途切れリンク'}</span>
          </span>
        </div>
      `;
    }

    function renderMIPs(datasetName, frameIdx) {
      const container = document.getElementById('mips-container');
      const frameStr = String(frameIdx).padStart(3, '0');
      container.innerHTML = `
        <div class="image-box">
          <div>XY Projection (Z-MIP)</div>
          <img src="viewer_data/${datasetName}/mips/frame_${frameStr}_xy.png" onerror="this.src='data:image/svg+xml;utf8,<svg xmlns=\'http://www.w3.org/2000/svg\' width=\'200\' height=\'200\'><rect width=\'200\' height=\'200\' fill=\'%23333\'/><text x=\'50%\' y=\'50%\' fill=\'%23888\' text-anchor=\'middle\'>No MIP Image</text></svg>'">
        </div>
        <div class="image-box">
          <div>XZ Projection (Y-MIP)</div>
          <img src="viewer_data/${datasetName}/mips/frame_${frameStr}_xz.png" onerror="this.src='data:image/svg+xml;utf8,<svg xmlns=\'http://www.w3.org/2000/svg\' width=\'200\' height=\'200\'><rect width=\'200\' height=\'200\' fill=\'%23333\'/><text x=\'50%\' y=\'50%\' fill=\'%23888\' text-anchor=\'middle\'>No MIP Image</text></svg>'">
        </div>
        <div class="image-box">
          <div>YZ Projection (X-MIP)</div>
          <img src="viewer_data/${datasetName}/mips/frame_${frameStr}_yz.png" onerror="this.src='data:image/svg+xml;utf8,<svg xmlns=\'http://www.w3.org/2000/svg\' width=\'200\' height=\'200\'><rect width=\'200\' height=\'200\' fill=\'%23333\'/><text x=\'50%\' y=\'50%\' fill=\'%23888\' text-anchor=\'middle\'>No MIP Image</text></svg>'">
        </div>
      `;
    }

    init();
  </script>
</body>
</html>
"""
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    print(f"Ensured index.html template at {html_path}")

def push_to_github_pages(working_dir: Path, repo_url: str, branch: str = 'gh-pages', token: str = '', enabled: bool = True):
    """
    GitHub Pages ブランチ (gh-pages) へ浅いクローン (--depth 1) を用いて index.html と viewer_data フォルダをデプロイ・プッシュする関数。
    index.html および viewer_data/ を gh-pages ブランチのルート直下に配置します。
    """
    if not enabled:
        print("PUSH_TO_GITHUB is False. Skipping GitHub Pages deployment.")
        return
        
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
            
    if not token:
        raise ValueError("GITHUB_TOKEN is missing or empty. Cannot push to GitHub without authentication token.")

    print(f"Syncing visualization artifacts to GitHub branch '{branch}'...")
    authed_repo_url = repo_url.replace('https://', f'https://x-access-token:{token}@')

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_repo = Path(tmp_dir) / 'repo'
        
        clone_cmd = f"git clone --branch {branch} --depth 1 {authed_repo_url} \"{tmp_repo}\""
        ret = os.system(clone_cmd)
        if ret != 0:
            print(f"Branch {branch} not found or clone failed. Initializing new repo...")
            ret_init = os.system(f"git clone --depth 1 {authed_repo_url} \"{tmp_repo}\"")
            if ret_init != 0:
                raise RuntimeError(f"Failed to clone repository: {repo_url}")
            os.system(f"cd \"{tmp_repo}\" && git checkout -b {branch}")

        # index.html のコピー
        ensure_index_html(working_dir)
        src_html = working_dir / 'index.html'
        if src_html.exists():
            shutil.copy2(src_html, tmp_repo / 'index.html')

        src_viewer_data = working_dir / 'viewer_data'
        dst_viewer_data = tmp_repo / 'viewer_data'
        if src_viewer_data.exists():
            if dst_viewer_data.exists():
                shutil.rmtree(dst_viewer_data)
            shutil.copytree(src_viewer_data, dst_viewer_data)

        datasets = []
        if dst_viewer_data.exists():
            datasets = [d.name for d in dst_viewer_data.iterdir() if d.is_dir()]
        
        with open(dst_viewer_data / 'datasets.json', 'w', encoding='utf-8') as f:
            json.dump({'datasets': sorted(datasets)}, f, indent=2)

        push_cmds = (
            f"cd \"{tmp_repo}\" && "
            "git config user.name \"Kaggle-Bot\" && "
            "git config user.email \"bot@kaggle.com\" && "
            "git add . && "
            "git commit -m \"Auto-update GT-centric cell tracking viewer\" && "
            f"git push {authed_repo_url} {branch}:{branch}"
        )
        ret_push = os.system(push_cmds)
        if ret_push != 0:
            raise RuntimeError(f"git push to {repo_url} on branch '{branch}' failed with exit code {ret_push}.")

        print(f"Successfully pushed visualization artifacts to {repo_url} on branch '{branch}'!")


In [ ]:
push_to_github_pages(WORKING_DIR, GITHUB_REPO, BRANCH_NAME, GITHUB_TOKEN, PUSH_TO_GITHUB)


## 6. Pipeline Completed

GT-centric cell tracking visualization dataset export and GitHub Pages deployment successfully finished.